In [10]:
!uv pip install -e ../../hedyPET

Using Python 3.11.11 environment at: /homes/hinge/Projects/hedypet-streamlit/.venv
Resolved 72 packages in 738ms                                        
   Building hedypet @ file:///homes/hinge/Projects/hedyPET             
   Building hedypet @ file:///homes/hinge/Projects/hedyPET     
   Building hedypet @ file:///homes/hinge/Projects/hedyPET     
   Building hedypet @ file:///homes/hinge/Projects/hedyPET     
   Building hedypet @ file:///homes/hinge/Projects/hedyPET     
      Built hedypet @ file:///homes/hinge/Projects/hedyPET     
Prepared 1 package in 819ms                                              
Uninstalled 1 package in 4ms
Installed 1 package in 19ms file:///homes/hinge/Projects/hed
 ~ hedypet==0.1.0 (from file:///homes/hinge/Projects/hedyPET)


In [19]:
import nibabel as nib
from pathlib import Path
from tqdm import tqdm
from matplotlib import pyplot as plt
from parse import parse
import numpy as np
import os 
import pandas as pd 
from hedypet.utils import get_norm_consts, get_patient_metadata
from hedypet.utils import load_splits 


df = pd.read_pickle("/homes/hinge/Projects/hedyPET/src/hedypet/analysis/acstatPSF_means.pkl")
RAW = Path("/depict/data/hedit/raw/")
DERIVATIVES = Path("/depict/data/hedit/derivatives/pipeline-bodydyn/")

def filter_and_combine_regions(df,regions):
    group_cols = ["sub","erosion","task"]
    results = []

    for k, v in regions.items():
        mask = df.region.isin(v) if isinstance(v, list) else df.region == v
        filtered = df[mask]
        
        if isinstance(v, list):
            result = filtered.groupby(group_cols).apply(
                lambda x: pd.Series({
                    'mu': (x.mu * x.n).sum() / x.n.sum(),
                    'n': x.n.sum()
                })
            ).reset_index()
        else:
            result = filtered
        
        result['region'] = k
        results.append(result)
    df = pd.concat(results, ignore_index=True)
    return df.drop("ix",axis=1)

regions = {
    "Spleen":"spleen",
    "Kidneys": ["kidney_right","kidney_left"],
    "Aorta": "aorta",
    "Liver":"liver",
    "Stomach":"stomach",
    "Lungs": ['lung_upper_lobe_left', 'lung_lower_lobe_left','lung_upper_lobe_right', 'lung_middle_lobe_right','lung_lower_lobe_right'],
    'Colon':"colon",
    'Subcutaneous fat':"subcutaneous_fat",
    "Skeletal muscle":"skeletal_muscle",
    'Visceral fat':"visceral_fat",
    "Gray matter":['Left-Cerebral-Cortex', 'Right-Cerebral-Cortex'], 
    "White matter": ['Right-Cerebral-White-Matter' , 'Left-Cerebral-White-Matter'],
    "Brain":"brain",
    "Esophagus": "esophagus",
    "Trachea": "trachea",
    "Small intestine": ["small_bowel","duodenum"],
    "Skull": "skull",
    "Ribs": [f"rib_left_{i}" for i in range(1,13)] + [f"rib_right_{i}" for i in range(1,13)],
    "Urinary bladder": "Urinary bladder",
    "Gallbladder": "gallbladder",
    "Pancreas": "pancreas",
    "Heart": "heart"
}
df = filter_and_combine_regions(df,regions)

/tmp/ipykernel_3731913/3186631560.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result['region'] = k
/tmp/ipykernel_3731913/3186631560.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = filtered.groupby(group_cols).apply(
/tmp/ipykernel_3731913/3186631560.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

In [15]:
df

,mu,std,n,sub,task,erosion,region
0,665.961304,176.173401,51140.0,sub-000,ts_total,0,Spleen
1,683.393677,166.655411,43203.0,sub-000,ts_total,1,Spleen
2,578.497192,134.692490,23751.0,sub-001,ts_total,0,Spleen
3,595.689697,127.745293,19431.0,sub-001,ts_total,1,Spleen
4,758.457520,129.332718,26857.0,sub-002,ts_total,0,Spleen
...,...,...,...,...,...,...,...
4395,738.622925,122.277992,65854.0,sub-097,ts_total,1,Heart
4396,1827.495728,1888.232910,123427.0,sub-098,ts_total,0,Heart
4397,1878.141602,1927.568481,106725.0,sub-098,ts_total,1,Heart
4398,2324.908203,2254.507812,127175.0,sub-099,ts_total,0,Heart


In [16]:
df.to_pickle("means.pkl")

In [21]:
df_norm

,0,sub
0,None,sub-000
1,None,sub-001
2,None,sub-002
3,None,sub-003
4,None,sub-004
...,...,...
95,None,sub-095
96,None,sub-096
97,None,sub-097
98,None,sub-098


In [23]:
df_norm

,suv,sul_janma,sul_james,sul_decazes,suv_id,sul_id,participant_id,sex,age,height,weight,demographic-group,blanket,InjectedRadioactivity,sub
0,422.994993,546.442392,538.024408,562.213553,467.767037,662.655350,sub-000,M,28,1.840,83,M18-34,NO,35.108584,sub-000
1,353.941072,444.370833,439.041745,414.230464,402.461168,482.511661,sub-001,M,49,1.890,82,M35-49,YES,29.023168,sub-001
2,392.588241,605.354393,533.736067,546.103077,408.379493,637.015837,sub-002,F,29,1.760,70,F18-34,NO,27.481177,sub-002
3,440.573255,736.748891,665.230612,648.594159,456.417759,742.189351,sub-003,F,51,1.650,75,F50-69,NO,33.042994,sub-003
4,375.153222,612.334645,547.835463,545.632795,370.349796,603.653999,sub-004,F,25,1.720,77,F18-34,YES,28.886798,sub-004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,419.565425,681.342408,608.542561,566.559644,430.462212,682.128952,sub-095,F,43,1.850,88,F35-49,YES,36.921757,sub-095
96,440.037638,671.210596,590.744451,588.535606,416.095971,601.124211,sub-096,F,77,1.680,62,F70-99,YES,27.282334,sub-096
97,401.503532,610.390078,536.950949,583.826855,406.040331,639.988465,sub-097,F,26,1.660,60,F18-34,YES,24.090212,sub-097
98,331.407197,445.285416,437.353174,462.673129,318.992615,474.909290,sub-098,M,24,1.865,93,M18-34,NO,30.820869,sub-098


In [24]:
import pandas as pd

data = []
for sub in load_splits()["all"]:
    d = get_norm_consts(sub)
    d.update(get_patient_metadata(sub)) 
    d["sub"] = sub
    d["demographic-group"] = d["demographic-group"][1:]
    data.append(d)

df_norm = pd.DataFrame(data)
df_norm.to_pickle("norm2.pkl")

In [25]:
df_norm

,suv,sul_janma,sul_james,sul_decazes,suv_id,sul_id,participant_id,sex,age,height,weight,demographic-group,blanket,InjectedRadioactivity,sub
0,422.994993,546.442392,538.024408,562.213553,467.767037,662.655350,sub-000,M,28,1.840,83,18-34,NO,35.108584,sub-000
1,353.941072,444.370833,439.041745,414.230464,402.461168,482.511661,sub-001,M,49,1.890,82,35-49,YES,29.023168,sub-001
2,392.588241,605.354393,533.736067,546.103077,408.379493,637.015837,sub-002,F,29,1.760,70,18-34,NO,27.481177,sub-002
3,440.573255,736.748891,665.230612,648.594159,456.417759,742.189351,sub-003,F,51,1.650,75,50-69,NO,33.042994,sub-003
4,375.153222,612.334645,547.835463,545.632795,370.349796,603.653999,sub-004,F,25,1.720,77,18-34,YES,28.886798,sub-004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,419.565425,681.342408,608.542561,566.559644,430.462212,682.128952,sub-095,F,43,1.850,88,35-49,YES,36.921757,sub-095
96,440.037638,671.210596,590.744451,588.535606,416.095971,601.124211,sub-096,F,77,1.680,62,70-99,YES,27.282334,sub-096
97,401.503532,610.390078,536.950949,583.826855,406.040331,639.988465,sub-097,F,26,1.660,60,18-34,YES,24.090212,sub-097
98,331.407197,445.285416,437.353174,462.673129,318.992615,474.909290,sub-098,M,24,1.865,93,18-34,NO,30.820869,sub-098
